# PyTorch Tutorial 47: Making Models Smaller

**Author:** PyTorch Tutorial Series  
**Date:** 2026  
**Prerequisites:** Notebook 46 (Edge ML Fundamentals) + Notebooks 00-05  
**Time:** ~1.5 hours

---

## What You'll Learn

In Notebook 46, we saw that models need to be small to run on edge devices. But what if you have a great big model and want to make it smaller? There are three main techniques:

1. **Pruning** — removing unnecessary weights (like trimming dead branches from a tree)
2. **Knowledge Distillation** — training a small model to mimic a big one (like a student learning from a teacher)
3. **Quantization** — using less precise numbers (like rounding 3.14159 to 3.14)

We'll implement all three from scratch and combine them into a full compression pipeline.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import copy
import os
import tempfile

print("PyTorch version:", torch.__version__)

## Setup: A Simple Model and Dataset

We'll use MNIST (handwritten digits) — it's small, fast to train, and perfect for learning.

First, let's define two models:
- **TeacherCNN**: A bigger, more accurate model
- **StudentCNN**: A smaller model we'll try to make just as good

In [ ]:
class TeacherCNN(nn.Module):
    """Bigger model — our 'teacher' for distillation."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 7 * 7, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class StudentCNN(nn.Module):
    """Smaller model — our 'student' that will learn from the teacher."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 7 * 7, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


# Compare sizes
teacher = TeacherCNN()
student = StudentCNN()

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

print(f"Teacher: {teacher_params:,} parameters ({teacher_params * 4 / 1024:.1f} KB)")
print(f"Student: {student_params:,} parameters ({student_params * 4 / 1024:.1f} KB)")
print(f"Student is {teacher_params / student_params:.1f}x smaller!")

In [ ]:
# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples:     {len(test_dataset):,}")

In [ ]:
def train_model(model, train_loader, epochs=3, lr=0.001):
    """Train a model on MNIST. Returns the trained model."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        acc = 100.0 * correct / total
        avg_loss = total_loss / len(train_loader)
        print(f"  Epoch {epoch + 1}/{epochs} - Loss: {avg_loss:.4f}, Acc: {acc:.1f}%")

    return model


def test_accuracy(model, test_loader):
    """Measure model accuracy on test set."""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    return 100.0 * correct / total

In [ ]:
# Train the teacher model first
print("Training Teacher Model...")
teacher = TeacherCNN()
teacher = train_model(teacher, train_loader, epochs=3)
teacher_acc = test_accuracy(teacher, test_loader)
print(f"\nTeacher Test Accuracy: {teacher_acc:.1f}%")

## 1. Pruning: Removing Unnecessary Weights

### The Idea

Not all weights in a neural network are equally important. Many weights are very close to zero — they barely contribute to the output. **Pruning** sets these unimportant weights to exactly zero.

Think of it like a garden: you trim the dead branches so the tree can focus its energy on the healthy ones.

### Why It Works

Research has shown that typical neural networks are **overparameterized** — they have way more parameters than they need. You can often remove 50-90% of weights with minimal accuracy loss!

In [ ]:
# Let's look at the weights of our trained teacher
# How many are already close to zero?
first_conv = teacher.features[0]  # First conv layer
weights = first_conv.weight.data.flatten()

total_weights = weights.numel()
near_zero = (weights.abs() < 0.01).sum().item()

print(f"First conv layer: {total_weights} weights")
print(f"Near zero (|w| < 0.01): {near_zero} ({100 * near_zero / total_weights:.1f}%)")
print(f"\nWeight statistics:")
print(f"  Min:  {weights.min().item():.4f}")
print(f"  Max:  {weights.max().item():.4f}")
print(f"  Mean: {weights.mean().item():.4f}")
print(f"  Std:  {weights.std().item():.4f}")

### Step-by-Step Pruning

PyTorch has built-in pruning tools. Let's prune our teacher model at different levels and see how accuracy changes.

In [ ]:
def prune_model(model, amount):
    """
    Prune a model by removing the smallest weights.
    amount: fraction of weights to remove (0.0 to 1.0)
    """
    pruned_model = copy.deepcopy(model)

    for name, module in pruned_model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            prune.l1_unstructured(module, name="weight", amount=amount)
            prune.remove(module, "weight")

    return pruned_model


def count_zeros(model):
    """Count the number of zero weights in a model."""
    total = 0
    zeros = 0
    for p in model.parameters():
        total += p.numel()
        zeros += (p == 0).sum().item()
    return zeros, total


# Try different pruning levels
print(f"{'Pruning %':<12} {'Zeros':>10} {'Sparsity':>10} {'Accuracy':>10}")
print("-" * 45)

for prune_pct in [0.0, 0.1, 0.3, 0.5, 0.7, 0.9]:
    pruned = prune_model(teacher, prune_pct)
    zeros, total = count_zeros(pruned)
    acc = test_accuracy(pruned, test_loader)
    sparsity = 100.0 * zeros / total
    print(f"{prune_pct * 100:>8.0f}%   {zeros:>10,} {sparsity:>8.1f}% {acc:>8.1f}%")

**What did we learn?**
- You can often prune 50% of weights with almost no accuracy loss
- At 70% pruning, accuracy starts dropping noticeably
- At 90% pruning, the model is much worse

The sweet spot depends on the model and task. Usually 30-70% works well.

> **Note:** Pruning alone doesn't make the model file smaller (zeros still take space). But it helps with quantization and specialized hardware that can skip zero-multiplications.

## 2. Knowledge Distillation: Learning From a Teacher

### The Idea

Instead of training a small model from scratch, we can train it to **mimic a bigger, smarter model**.

**Analogy:** Imagine a professor (teacher) explaining a topic. A textbook just says "the answer is A." But the professor says "it's probably A, could be B, definitely not C or D." This richer information helps students learn faster.

### How It Works

1. Train a big **teacher** model (we already did this)
2. The teacher produces **soft labels** — probabilities for each class (not just the answer)
3. Train a small **student** model using both:
   - The real labels (hard targets)
   - The teacher's soft labels (soft targets)

### Temperature Scaling

The teacher's output is usually very confident (e.g., 99% for the right answer). We use a **temperature** parameter to "soften" the probabilities, revealing the relationships between classes.

- Temperature = 1: normal softmax (very peaked)
- Temperature = 5: softer probabilities (more information)

In [ ]:
# Demonstrate temperature scaling
logits = torch.tensor([5.0, 2.0, 0.5, -1.0, -3.0])  # Example model output

print("Effect of temperature on softmax probabilities:")
print(f"\n{'Class':<8}", end="")
for t in [1, 2, 5, 10]:
    print(f"{'T=' + str(t):>10}", end="")
print()
print("-" * 50)

for i in range(5):
    print(f"Class {i}:", end="")
    for t in [1, 2, 5, 10]:
        prob = F.softmax(logits / t, dim=0)[i].item()
        print(f"{prob:>10.3f}", end="")
    print()

print("\nAt T=1, Class 0 dominates. At T=10, probabilities are spread out.")
print("Higher temperature reveals more about the relationships between classes.")

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, temperature=5.0, alpha=0.5):
    """
    Combined loss for knowledge distillation.

    Two parts:
    1. Soft loss: student tries to match teacher's soft probabilities
    2. Hard loss: student also learns from the true labels

    alpha controls the balance (0.5 = equal weight).
    """
    # Soft targets from teacher (with temperature)
    soft_teacher = F.softmax(teacher_logits / temperature, dim=1)
    soft_student = F.log_softmax(student_logits / temperature, dim=1)

    # KL divergence between soft distributions
    # Multiply by T^2 to keep gradients at similar scale
    soft_loss = F.kl_div(soft_student, soft_teacher, reduction="batchmean")
    soft_loss = soft_loss * (temperature ** 2)

    # Hard loss (regular cross-entropy with true labels)
    hard_loss = F.cross_entropy(student_logits, labels)

    # Combine both losses
    return alpha * soft_loss + (1 - alpha) * hard_loss


def train_with_distillation(student, teacher, train_loader, epochs=3, temperature=5.0):
    """Train a student model using knowledge distillation from a teacher."""
    optimizer = torch.optim.Adam(student.parameters(), lr=0.001)
    teacher.eval()  # Teacher is frozen!
    student.train()

    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            with torch.no_grad():
                teacher_logits = teacher(images)

            student_logits = student(images)
            loss = distillation_loss(
                student_logits, teacher_logits, labels, temperature=temperature
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = student_logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        acc = 100.0 * correct / total
        avg_loss = total_loss / len(train_loader)
        print(f"  Epoch {epoch + 1}/{epochs} - Loss: {avg_loss:.4f}, Acc: {acc:.1f}%")

    return student

In [ ]:
# Train student WITHOUT distillation (baseline)
print("Training Student WITHOUT teacher (baseline)...")
student_alone = StudentCNN()
student_alone = train_model(student_alone, train_loader, epochs=3)
alone_acc = test_accuracy(student_alone, test_loader)
print(f"Student (alone) Accuracy: {alone_acc:.1f}%")

print("\n" + "=" * 50)

# Train student WITH distillation
print("\nTraining Student WITH teacher (distillation)...")
student_distilled = StudentCNN()
student_distilled = train_with_distillation(
    student_distilled, teacher, train_loader, epochs=3
)
distilled_acc = test_accuracy(student_distilled, test_loader)
print(f"Student (distilled) Accuracy: {distilled_acc:.1f}%")

print("\n" + "=" * 50)
print(f"\nResults:")
print(f"  Teacher accuracy:            {teacher_acc:.1f}%")
print(f"  Student (trained alone):     {alone_acc:.1f}%")
print(f"  Student (with distillation): {distilled_acc:.1f}%")
print(f"  Improvement from distill:    +{distilled_acc - alone_acc:.1f}%")

**What did we learn?**
- The distilled student is **closer to the teacher's accuracy** than a student trained alone
- The student is still much smaller than the teacher
- The "soft labels" from the teacher provide richer learning signal

This technique is used everywhere in production: OpenAI, Google, and Meta all use distillation to create smaller, faster models.

## 3. Quantization: Using Less Precise Numbers

### The Idea

Neural network weights are stored as **32-bit floats** (FP32). But for inference, we often don't need that much precision.

**Analogy:** If someone asks your height, you say "5 foot 10" — not "5 feet, 10 inches, and 3.7 millimeters." The rounded version is good enough.

Quantization converts FP32 (4 bytes) to INT8 (1 byte) = **4x smaller model**.

> Notebook 20 covers quantization theory in depth. Here we'll focus on the **practical API** — making it work in 3 lines of code.

In [ ]:
def get_model_file_size(model):
    """Save model to a temp file and measure actual file size."""
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pt") as f:
        torch.save(model.state_dict(), f.name)
        size_bytes = os.path.getsize(f.name)
    os.unlink(f.name)
    return size_bytes


# Original model size
original_size = get_model_file_size(student_distilled)
print(f"Original student model size: {original_size / 1024:.1f} KB")

# Quantize with dynamic quantization (easiest method — just 1 line!)
quantized_student = torch.ao.quantization.quantize_dynamic(
    student_distilled, {nn.Linear}, dtype=torch.qint8
)

# Quantized model size
quantized_size = get_model_file_size(quantized_student)
quantized_acc = test_accuracy(quantized_student, test_loader)

print(f"Quantized model size:        {quantized_size / 1024:.1f} KB")
print(f"Compression ratio:           {original_size / quantized_size:.1f}x smaller")
print(f"\nAccuracy before:             {distilled_acc:.1f}%")
print(f"Accuracy after quantization: {quantized_acc:.1f}%")
print(f"Accuracy change:             {quantized_acc - distilled_acc:+.1f}%")

### Types of Quantization

| Method | When | Effort | Quality |
|--------|------|--------|----------|
| **Dynamic Quantization** | After training | 1 line of code | Good |
| **Static Quantization** | After training (needs calibration data) | ~10 lines | Better |
| **Quantization-Aware Training (QAT)** | During training | Moderate | Best |

We used dynamic quantization above — it's the easiest and works well for most cases.

## 4. The Full Pipeline: Combine Everything

Now let's combine all three techniques and see the cumulative effect.

Pipeline: **Train Teacher** -> **Distill to Student** -> **Prune Student** -> **Quantize Student**

In [ ]:
# Step 1: We already have a trained teacher
print("=== Full Compression Pipeline ===")
print()

# Step 2: Distill (already done)
print("Step 1: Knowledge Distillation")
print(f"  Teacher:  {teacher_params:,} params, Acc: {teacher_acc:.1f}%")
print(f"  Student:  {student_params:,} params, Acc: {distilled_acc:.1f}%")
print(f"  Size reduction: {teacher_params / student_params:.1f}x")

# Step 3: Prune the student
print("\nStep 2: Pruning (50%)")
pruned_student = prune_model(student_distilled, amount=0.5)
pruned_acc = test_accuracy(pruned_student, test_loader)
zeros, total = count_zeros(pruned_student)
print(f"  Zeros: {zeros:,} / {total:,} ({100 * zeros / total:.1f}% sparse)")
print(f"  Accuracy: {pruned_acc:.1f}%")

# Step 4: Quantize the pruned student
print("\nStep 3: Quantization (INT8)")
final_model = torch.ao.quantization.quantize_dynamic(
    pruned_student, {nn.Linear}, dtype=torch.qint8
)
final_acc = test_accuracy(final_model, test_loader)
final_size = get_model_file_size(final_model)
teacher_size = get_model_file_size(teacher)

print(f"  Final accuracy: {final_acc:.1f}%")
print(f"  Final size:     {final_size / 1024:.1f} KB")

In [ ]:
# Summary table
print("\n" + "=" * 65)
print("COMPRESSION PIPELINE SUMMARY")
print("=" * 65)
print(f"{'Stage':<30} {'Size (KB)':>12} {'Accuracy':>10} {'vs Teacher':>12}")
print("-" * 65)

teacher_kb = teacher_size / 1024
original_kb = original_size / 1024
final_kb = final_size / 1024

print(f"{'Teacher (original)':<30} {teacher_kb:>10.1f} {teacher_acc:>9.1f}% {'baseline':>12}")
print(f"{'After Distillation':<30} {original_kb:>10.1f} {distilled_acc:>9.1f}% {original_size / teacher_size:>10.1f}x")
print(f"{'After Prune + Quantize':<30} {final_kb:>10.1f} {final_acc:>9.1f}% {final_size / teacher_size:>10.1f}x")
print("-" * 65)
print(f"Total compression: {teacher_size / final_size:.1f}x smaller")
print(f"Accuracy retained: {final_acc:.1f}% (teacher was {teacher_acc:.1f}%)")

## 5. Try It Yourself!

**Exercise 1:** Try different pruning percentages (20%, 40%, 60%, 80%) on the distilled student. At what point does accuracy drop significantly?

**Exercise 2:** Try different distillation temperatures (1, 3, 5, 10, 20). Which gives the best student accuracy?

**Exercise 3:** What happens if you prune the teacher BEFORE distillation? Does the student still learn well from a pruned teacher?

## 6. Recap

We learned three ways to make models smaller:

| Technique | What it does | Typical reduction | Accuracy impact |
|-----------|-------------|-------------------|------------------|
| **Pruning** | Sets small weights to zero | 2-5x fewer active weights | Small |
| **Distillation** | Small model mimics big model | 5-20x fewer parameters | Small |
| **Quantization** | FP32 to INT8 | 4x smaller files | Minimal |

**Combined**, these can shrink a model by **10-50x** with only 1-3% accuracy loss.

**The standard production pipeline is:** Train big -> Distill to small -> Prune -> Quantize

**What's next:** In Notebook 48, we'll learn how to **export** these compressed models so they can actually run on phones, Raspberry Pi, and other edge devices.